# Silver Layer — Orchestration
Run all Silver transformation notebooks in sequence. Single entry point for the Silver layer.

## Setup Connection

In [1]:
import os
from dotenv import load_dotenv
from clickzetta.zettapark.session import Session
from clickzetta.zettapark import functions as F
from clickzetta.zettapark.types import StringType, DateType

load_dotenv()
session = Session.builder.configs({
    "username":  os.environ["CLICKZETTA_USERNAME"],
    "password":  os.environ["CLICKZETTA_PASSWORD"],
    "service":   os.environ["CLICKZETTA_SERVICE"],
    "instance":  os.environ["CLICKZETTA_INSTANCE"],
    "workspace": os.environ["CLICKZETTA_WORKSPACE"],
    "schema":    "silver",
    "vcluster":  os.environ["CLICKZETTA_VCLUSTER"],
}).create()

def _trim(df):
    for field in df.schema.fields:
        if isinstance(field.datatype, StringType):
            df = df.with_column(field.name, F.trim(F.col(field.name)))
    return df

## CRM — crm_customers

In [2]:
df = _trim(session.table(f"bronze.crm_cust_info"))
df = (df
    .with_column("cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status")) == "S", "Single")
         .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
         .otherwise("n/a"))
    .with_column("cst_gndr",
        F.when(F.upper(F.col("cst_gndr")) == "F", "Female")
         .when(F.upper(F.col("cst_gndr")) == "M", "Male")
         .otherwise("n/a"))
    .filter(F.col("cst_id").is_not_null())
)
for old, new in {"cst_id":"customer_id","cst_key":"customer_number","cst_firstname":"first_name",
                 "cst_lastname":"last_name","cst_marital_status":"marital_status",
                 "cst_gndr":"gender","cst_create_date":"created_date"}.items():
    df = df.with_column_renamed(old, new)
df.write.save_as_table(f"silver.crm_customers", mode="overwrite")
print("crm_customers OK")

crm_customers OK


## CRM — crm_products

In [3]:
df = _trim(session.table(f"bronze.crm_prd_info"))
df = (df
    .with_column("cat_id", F.regexp_replace(F.substring(F.col("prd_key"), 1, 5), F.lit("-"), F.lit("_")))
    .with_column("prd_key", F.substring(F.col("prd_key"), 7, F.length(F.col("prd_key"))))
    .with_column("prd_cost", F.coalesce(F.col("prd_cost"), F.lit(0)))
    .with_column("prd_line",
        F.when(F.upper(F.col("prd_line")) == "M", "Mountain")
         .when(F.upper(F.col("prd_line")) == "R", "Road")
         .when(F.upper(F.col("prd_line")) == "S", "Other Sales")
         .when(F.upper(F.col("prd_line")) == "T", "Touring")
         .otherwise("n/a"))
    .with_column("prd_start_dt", F.col("prd_start_dt").cast(DateType()))
)
for old, new in {"prd_id":"product_id","cat_id":"category_id","prd_key":"product_number",
                 "prd_nm":"product_name","prd_cost":"product_cost","prd_line":"product_line",
                 "prd_start_dt":"start_date","prd_end_dt":"end_date"}.items():
    df = df.with_column_renamed(old, new)
df.write.save_as_table(f"silver.crm_products", mode="overwrite")
print("crm_products OK")

crm_products OK


## CRM — crm_sales

In [4]:
df = _trim(session.table(f"bronze.crm_sales_details"))
for date_col in ["sls_order_dt", "sls_ship_dt", "sls_due_dt"]:
    df = df.with_column(date_col,
        F.when((F.col(date_col) == 0) | (F.length(F.col(date_col).cast("string")) != 8),
               F.lit(None).cast(DateType()))
         .otherwise(F.to_date(F.col(date_col).cast("string"), "yyyyMMdd")))
df = df.with_column("sls_price",
    F.when(F.col("sls_price").is_null() | (F.col("sls_price") <= 0),
           F.when(F.col("sls_quantity") != 0, F.col("sls_sales") / F.col("sls_quantity"))
            .otherwise(F.lit(None)))
     .otherwise(F.col("sls_price")))
for old, new in {"sls_ord_num":"order_number","sls_prd_key":"product_number","sls_cust_id":"customer_id",
                 "sls_order_dt":"order_date","sls_ship_dt":"ship_date","sls_due_dt":"due_date",
                 "sls_sales":"sales_amount","sls_quantity":"quantity","sls_price":"price"}.items():
    df = df.with_column_renamed(old, new)
df.write.save_as_table(f"silver.crm_sales", mode="overwrite")
print("crm_sales OK")

crm_sales OK


## ERP — erp_customers

In [5]:
df = _trim(session.table(f"bronze.erp_cust_az12"))
df = (df
    .with_column("cid",
        F.when(F.col("cid").startswith("NAS"), F.substring(F.col("cid"), 4, F.length(F.col("cid"))))
         .otherwise(F.col("cid")))
    .with_column("bdate",
        F.when(F.col("bdate") > F.current_date(), F.lit(None)).otherwise(F.col("bdate")))
    .with_column("gen",
        F.when(F.upper(F.col("gen")).isin("F", "FEMALE"), "Female")
         .when(F.upper(F.col("gen")).isin("M", "MALE"), "Male")
         .otherwise("n/a"))
)
for old, new in {"cid":"customer_number","bdate":"birth_date","gen":"gender"}.items():
    df = df.with_column_renamed(old, new)
df.write.save_as_table(f"silver.erp_customers", mode="overwrite")
print("erp_customers OK")

erp_customers OK


## ERP — erp_customer_location

In [6]:
df = _trim(session.table(f"bronze.erp_loc_a101"))
df = (df
    .with_column("cid", F.regexp_replace(F.col("cid"), F.lit("-"), F.lit("")))
    .with_column("cntry",
        F.when(F.col("cntry") == "DE", "Germany")
         .when(F.col("cntry").isin("US", "USA"), "United States")
         .when(F.col("cntry").is_null() | (F.col("cntry") == ""), "n/a")
         .otherwise(F.col("cntry")))
)
for old, new in {"cid":"customer_number","cntry":"country"}.items():
    df = df.with_column_renamed(old, new)
df.write.save_as_table(f"silver.erp_customer_location", mode="overwrite")
print("erp_customer_location OK")

erp_customer_location OK


## ERP — erp_product_category

In [7]:
df = _trim(session.table(f"bronze.erp_px_cat_g1v2"))
df = df.with_column("maintenance",
    F.when(F.upper(F.col("maintenance")) == "YES", F.lit(True))
     .when(F.upper(F.col("maintenance")) == "NO", F.lit(False))
     .otherwise(F.lit(None)))
for old, new in {"id":"category_id","cat":"category","subcat":"subcategory","maintenance":"maintenance_flag"}.items():
    df = df.with_column_renamed(old, new)
df.write.save_as_table(f"silver.erp_product_category", mode="overwrite")
print("erp_product_category OK")

erp_product_category OK
